# Análise do Índice Composto de Risco (ICR)

Notebook de apoio ao TCC — explora os resultados gerados pelos scripts
`01_coletar.py`, `02_construir_icr.py`, `03_validar_explicar.py` e
`04_validar_acoes.py`. Rode-os antes (ou os dados em `data/processed/` já
precisam existir).

Dois painéis: **prudencial** (com Basileia; cooperativas/bancos médios) e
**sistêmico** (instituições individuais; inclui os grandes bancos).

In [ ]:
import sys; sys.path.insert(0, '../src')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from icr.config import DIR_PROCESSED, DIR_RAW, carregar_config, dir_processed_painel
from icr import validacao, explicabilidade as expl
cfg = carregar_config()
pd.set_option('display.max_columns', 30)
paineis = [p['nome'] for p in cfg['paineis']]
icr = {n: pd.read_parquet(dir_processed_painel(n)/'icr.parquet') for n in paineis}
{n: (d.shape[0], int(d.AnoMes.nunique())) for n, d in icr.items()}

## 1. Ranking de risco — trimestre mais recente (painel sistêmico)

In [ ]:
d = icr['sistemico']; ult = int(d.AnoMes.max())
rk = d[d.AnoMes == ult].sort_values('ICR', ascending=False)
cols = ['nome','ICR','faixa','roe','liquidez','eficiencia','ativo_total_mil']
print(f'Trimestre {ult} — TOP 10 mais sólidos')
display(rk[cols].head(10))
print('5 piores (zona de risco)')
display(rk[cols].tail(5))

## 2. Distribuição dos indicadores (último trimestre, sistêmico)

In [ ]:
inds = [c for c in ['roa','roe','liquidez','eficiencia','alavancagem','basileia'] if c in d.columns and d[c].notna().any()]
dult = d[d.AnoMes == ult]
fig, axes = plt.subplots(1, len(inds), figsize=(3.2*len(inds), 3))
for ax, c in zip(np.atleast_1d(axes), inds):
    s = dult[c].clip(dult[c].quantile(.01), dult[c].quantile(.99))
    ax.hist(s.dropna(), bins=30, color='steelblue'); ax.set_title(c)
plt.tight_layout()

## 3. Evolução temporal do ICR do sistema (comparação entre painéis)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for n in paineis:
    s = validacao.serie_sistema(icr[n])
    x = [f'{a//100}T{(a%100)//3}' for a in s.AnoMes]
    ax.plot(x, s.ICR_medio_ponderado, marker='o', label=n)
ax.axhline(0, color='grey', lw=.8); ax.legend(); ax.tick_params(axis='x', rotation=90)
ax.set_title('ICR do sistema (pond. por ativo)'); ax.set_ylabel('ICR')

## 4. Validação macroeconômica (ICR x séries do SGS)

In [ ]:
macro = pd.read_parquet(DIR_RAW/'sgs_trimestral.parquet')
s = validacao.serie_sistema(icr['sistemico'])
corr = validacao.correlacao_macro(s, macro)
display(corr)
fig, ax = plt.subplots(figsize=(7,4))
ax.barh(corr.serie_macro, corr.pearson, color='tab:red')
ax.axvline(0, color='grey'); ax.set_title('Correlação de Pearson: ICR sistêmico x SGS')

## 5. Explicabilidade — o que mais move o ICR

In [ ]:
imp = expl.importancia_por_contribuicao(icr['sistemico'], cfg['indicadores'])
display(imp)
_, shap_resumo = expl.shap_zona_risco(icr['sistemico'], cfg['indicadores'])
display(shap_resumo)

## 6. Validação com ações (B3)

In [ ]:
p = DIR_PROCESSED/'validacao_acoes.csv'
display(pd.read_csv(p)) if p.exists() else print('Rode scripts/04_validar_acoes.py primeiro.')